# Vision Guard - Colab Run

Run the project directly from GitHub, launch the Vision Guard Gradio UI, scan the video first, then search with natural language and review matched frames before exporting clips.

## 1. clone repo

In [ ]:
import os
if not os.path.exists('visionguard-ai'):
    !git clone https://github.com/priyansupattanaik/visionguard-ai.git
%cd visionguard-ai
!git pull

## 2. mount drive for cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2A. set persistent model cache

## 2B. optional Hugging Face token

If you store `HF_TOKEN` in Colab secrets, this cell loads it and removes unauthenticated Hub warnings for model downloads.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab secrets')
    else:
        print('HF_TOKEN secret not set; downloads will remain unauthenticated')
except Exception:
    print('Colab secrets not available; set os.environ["HF_TOKEN"] manually if needed')

In [ ]:
import os
base = '/content/drive/MyDrive/visionguard_cache'
paths = {
    'HF_HOME': f'{base}/hf',
    'TRANSFORMERS_CACHE': f'{base}/hf/transformers',
    'HUGGINGFACE_HUB_CACHE': f'{base}/hf/hub',
    'TORCH_HOME': f'{base}/torch',
    'YOLO_CONFIG_DIR': f'{base}/ultralytics',
    'ULTRALYTICS_SETTINGS': f'{base}/ultralytics/settings.json',
}
for key, value in paths.items():
    os.environ[key] = value
for key in ['HF_HOME', 'TRANSFORMERS_CACHE', 'HUGGINGFACE_HUB_CACHE', 'TORCH_HOME', 'YOLO_CONFIG_DIR']:
    os.makedirs(os.environ[key], exist_ok=True)
print('model cache:', base)

## 3. install deps

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. optional gpu check

In [ ]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'

## 5. launch app in Colab

This path starts the Gradio app in the background, waits until port 7860 is actually reachable, and then opens it with Colab's port window helper.

In [ ]:
import os
import subprocess
import time
import urllib.request
from google.colab import output

os.environ['VISION_GUARD_HOST'] = '0.0.0.0'
os.environ['GRADIO_SHARE'] = '0'

proc = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

deadline = time.time() + 300
ready = False
while time.time() < deadline:
    line = proc.stdout.readline()
    if line:
        print(line, end='')
    try:
        with urllib.request.urlopen('http://127.0.0.1:7860/', timeout=3) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    raise RuntimeError('Vision Guard did not become reachable on port 7860. Check the logs above.')

print('\nApp Ready')
output.serve_kernel_port_as_window(7860)

## 5A. optional live logs

In [ ]:
while True:
    line = proc.stdout.readline()
    if not line:
        break
    print(line, end='')

## 6. use the app

1. upload a CCTV video or use sample assets
2. click `step 1: scan video`
3. watch the live indexing preview until scan completes
4. type a query like `person sitting near gate`, `fight near road`, `car accident`, `white car entering`, `yellow car`
5. click `step 2: find matches`
6. review the matched-frame gallery and timestamp table
7. export only the clips you choose